# Stage 6 — Inference & Counterfactual Warming Analysis

Demonstrates the `Predictor` class that the FastAPI backend imports directly.

## What this notebook covers

1. Load all model artefacts via `Predictor.load()`
2. `GET /health` and `GET /species` endpoint equivalents
3. `POST /predict` — per-observation species predictions
4. `POST /predict_grid` — spatial grid predictions with counterfactual temperature offsets (+0, +1, +2, +3 °C)
5. Counterfactual summary: how species richness and individual species shift under warming
6. API response JSON examples (for your partner's frontend)

**Note:** Run `04_evaluation.ipynb` first to generate `species_labels_filtered.json` (Stage 5 spatial CV).  
The predictor works without it but all species are treated as high-confidence.

In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = _here.parent if _here.name == 'notebooks' else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)-8s %(name)s  %(message)s',
    force=True,
)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from pipeline.inference import Predictor
from config import (
    SAMPLED_PATH, MODELS_DIR,
    SD_LAT_MIN, SD_LAT_MAX, SD_LON_MIN, SD_LON_MAX,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110})
print('Setup complete')

## 1. Load artefacts — `Predictor.load()`

In [ ]:
predictor = Predictor.load()
print('Predictor loaded successfully')

## 2. `GET /health`

In [ ]:
health = predictor.health()
print(json.dumps(health, indent=2))

## 3. `GET /species`

In [ ]:
species_list = predictor.list_species()
print(f'Total species: {len(species_list)}')
high_conf = [s for s in species_list if s['high_confidence']]
print(f'High-confidence (spatial CV AUC ≥ threshold): {len(high_conf)}')
print()
print('First 5 entries:')
print(json.dumps(species_list[:5], indent=2))

## 4. `POST /predict` — per-observation predictions

In [ ]:
# Sample a few real observations from the dataset
df = pd.read_parquet(SAMPLED_PATH)
sample = df[df['presence'] == 1].sample(3, random_state=0)

observations = [
    {
        'temperature_c': float(row['temperature_c']),
        'humidity_pct':  float(row['humidity_pct']),
        'lat':           float(row['lat']),
        'lon':           float(row['lon']),
        'day_of_year':   int(row['day_of_year']),
        'taxon_name':    row['taxon_name'],   # extra field — passed through
    }
    for _, row in sample.iterrows()
]

results = predictor.predict(observations)

for i, (obs, res) in enumerate(zip(observations, results)):
    top5 = res['predictions'][:5]
    print(f'Observation {i+1}: lat={obs["lat"]:.4f}, lon={obs["lon"]:.4f}, '
          f'doy={obs["day_of_year"]}, taxon={obs["taxon_name"]}')
    print(f'  True species is in top-5: {obs["taxon_name"] in [p["taxon_name"] for p in top5]}')
    for p in top5:
        print(f'  {p["probability"]:.3f}  {p["taxon_name"]}  ({p["common_name"]})')
    print()

## 5. Counterfactual warming — `POST /predict_grid`

Run the same grid under baseline (+0°C), +1°C, +2°C, and +3°C offsets.  
All other features (humidity, lat, lon, day-of-year) are held constant.

In [ ]:
OFFSETS = [0.0, 1.0, 2.0, 3.0]
N_LAT, N_LON = 30, 30
DOY = 120  # late April — peak season

grid_results = {}
for offset in OFFSETS:
    grid_results[offset] = predictor.predict_grid(
        lat_min=SD_LAT_MIN, lat_max=SD_LAT_MAX,
        lon_min=SD_LON_MIN, lon_max=SD_LON_MAX,
        day_of_year=DOY,
        temperature_offset=offset,
        n_lat=N_LAT, n_lon=N_LON,
    )
    n_sp = len(grid_results[offset]['species'])
    print(f'offset={offset:+.0f}°C: grid {N_LAT}×{N_LON}, {n_sp} species')

In [ ]:
# Convert to numpy for analysis
def grid_to_array(g):
    """Return (n_lat, n_lon, S) float32 array of probabilities."""
    return np.array(g['probabilities'], dtype=np.float32)  # (n_lat, n_lon, S)

grids = {off: grid_to_array(grid_results[off]) for off in OFFSETS}

# Species richness (expected count above threshold = 0.5)
THRESHOLD = 0.5
richness = {off: (grids[off] >= THRESHOLD).sum(axis=2) for off in OFFSETS}

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
vmax = max(r.max() for r in richness.values())

for ax, offset in zip(axes, OFFSETS):
    g = grid_results[offset]
    lats = np.array(g['lats'])
    lons = np.array(g['lons'])
    r = richness[offset]
    
    im = ax.imshow(
        r, origin='lower', aspect='auto',
        extent=[lons[0], lons[-1], lats[0], lats[-1]],
        cmap='YlOrRd', vmin=0, vmax=vmax,
    )
    ax.set_title(f'T{offset:+.0f}°C  |  DOY={DOY}')
    ax.set_xlabel('Lon'); ax.set_ylabel('Lat')
    plt.colorbar(im, ax=ax, label='Species richness' if offset == OFFSETS[-1] else '')

plt.suptitle('Predicted species richness under warming scenarios (threshold=0.5)', fontsize=12)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'counterfactual_richness.png', bbox_inches='tight', dpi=120)
plt.show()

## 6. Per-species response to warming

In [ ]:
sp_names = grid_results[0.0]['species']
S = len(sp_names)

# Mean probability per species per scenario (averaged over grid cells)
mean_prob = {off: grids[off].mean(axis=(0, 1)) for off in OFFSETS}  # each shape (S,)

# Change from baseline
delta_1 = mean_prob[1.0] - mean_prob[0.0]
delta_2 = mean_prob[2.0] - mean_prob[0.0]
delta_3 = mean_prob[3.0] - mean_prob[0.0]

response_df = pd.DataFrame({
    'species': sp_names,
    'baseline': mean_prob[0.0],
    'delta_1c': delta_1,
    'delta_2c': delta_2,
    'delta_3c': delta_3,
}).sort_values('delta_3c')

n_losers  = (response_df['delta_3c'] < -0.02).sum()
n_gainers = (response_df['delta_3c'] >  0.02).sum()
print(f'At +3°C:')
print(f'  Species losing probability  (Δ < -0.02): {n_losers}')
print(f'  Species gaining probability (Δ > +0.02): {n_gainers}')
print(f'  Mostly unchanged (|Δ| ≤ 0.02)          : {S - n_losers - n_gainers}')

print(f'\nTop 5 losers at +3°C:')
for _, row in response_df.head(5).iterrows():
    print(f"  {row['delta_3c']:+.3f}  {row['species']}  (baseline={row['baseline']:.3f})")
print(f'\nTop 5 gainers at +3°C:')
for _, row in response_df.tail(5).iterrows():
    print(f"  {row['delta_3c']:+.3f}  {row['species']}  (baseline={row['baseline']:.3f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Change distribution at +3°C
axes[0].hist(delta_3, bins=25, color='steelblue', edgecolor='none', alpha=0.85)
axes[0].axvline(0, color='black', lw=1)
axes[0].axvline(-0.02, color='tomato',    ls='--', lw=1, label='Loss threshold')
axes[0].axvline( 0.02, color='darkgreen', ls='--', lw=1, label='Gain threshold')
axes[0].set_title('Distribution of mean-probability change at +3°C')
axes[0].set_xlabel('Δ mean probability (grid-averaged)')
axes[0].set_ylabel('Species count')
axes[0].legend()

# Warming trajectory for top 5 losers and gainers
top_losers  = response_df.head(5)['species'].tolist()
top_gainers = response_df.tail(5)['species'].tolist()

for sp in top_losers:
    j = sp_names.index(sp)
    vals = [mean_prob[off][j] for off in OFFSETS]
    axes[1].plot(OFFSETS, vals, 'o-', color='tomato', alpha=0.7, lw=1.5,
                 label=sp.split()[-1] if sp == top_losers[0] else None)
for sp in top_gainers:
    j = sp_names.index(sp)
    vals = [mean_prob[off][j] for off in OFFSETS]
    axes[1].plot(OFFSETS, vals, 'o-', color='darkgreen', alpha=0.7, lw=1.5,
                 label=sp.split()[-1] if sp == top_gainers[0] else None)

axes[1].set_title('Warming trajectories — top 5 losers (red) & gainers (green) at +3°C')
axes[1].set_xlabel('Temperature offset (°C)')
axes[1].set_ylabel('Mean predicted probability')
axes[1].set_xticks(OFFSETS)
axes[1].set_xticklabels([f'+{o:.0f}°C' for o in OFFSETS])

plt.suptitle('Counterfactual warming: species-level response', fontsize=12)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'counterfactual_species_response.png', bbox_inches='tight', dpi=120)
plt.show()

## 7. API response examples (for partner documentation)

In [ ]:
# /predict example
example_obs = [{
    'temperature_c': 18.5,
    'humidity_pct': 72.0,
    'lat': 32.871,
    'lon': -117.247,
    'day_of_year': 120,
}]

predict_result = predictor.predict(example_obs)
# Trim to top 5 predictions for display
predict_example = [
    {**r, 'predictions': r['predictions'][:5]}
    for r in predict_result
]

print('POST /predict example response:')
print(json.dumps(predict_example, indent=2))

In [ ]:
# /predict_grid mini example (3×3 grid, 3 species)
top3_species = [s['taxon_name'] for s in species_list[:3]]

grid_example = predictor.predict_grid(
    lat_min=32.86, lat_max=32.88,
    lon_min=-117.26, lon_max=-117.24,
    day_of_year=120,
    temperature_offset=2.0,
    n_lat=3, n_lon=3,
    species_filter=top3_species,
)

print('POST /predict_grid example response (3×3 grid, 3 species, +2°C):')
print(json.dumps(grid_example, indent=2))

In [ ]:
# /species example
species_example = predictor.list_species()[:3]
print('GET /species example response (first 3):')
print(json.dumps(species_example, indent=2))

## 8. Summary

In [ ]:
h = predictor.health()
richness_baseline = richness[0.0]
richness_3c       = richness[3.0]

sep = '=' * 64
lines = [
    '',
    sep,
    '  STAGE 6 — INFERENCE SUMMARY',
    sep,
    f'  Species in model             : {h["n_species"]}',
    f'  High-confidence species      : {h["n_high_confidence_species"]}',
    f'  Device                       : {h["device"]}',
    '',
    f'  Counterfactual grid ({N_LAT}×{N_LON}), DOY={DOY}:',
    f'    Mean richness baseline      : {richness_baseline.mean():.1f} species/cell',
    f'    Mean richness +3°C          : {richness_3c.mean():.1f} species/cell',
    f'    Richness change             : {richness_3c.mean()-richness_baseline.mean():+.1f}',
    f'    Losers (Δ < -0.02 mean prob): {n_losers} species',
    f'    Gainers (Δ > +0.02 mean prob): {n_gainers} species',
    '',
    '  Artefacts available for API:',
    '    models/sdm_model.pt',
    '    models/scaler.pkl',
    '    models/species_labels.json',
    '    models/species_metadata.json',
    '    models/species_labels_filtered.json  (after Stage 5)',
    sep,
    '',
    '  Pipeline complete. ✓',
    sep,
]
print('\n'.join(lines))